# Four Positive Charges: Collinear Chain Bound State — Live

This notebook demonstrates the first known **N $\geq$ 3 like-charge
sub-critical Weber bound state**: four equal positive charges oscillating
on a line, held together by Weber's velocity-dependent attraction below
the critical radius.

## Physics Background

### Weber's Velocity-Dependent Force

Weber's force law between two charges $q_i$, $q_j$ separated by distance
$r$ with radial velocity $\dot{r}$ and acceleration $\ddot{r}$ is

$$F = \frac{q_i q_j}{r^2}\left(1 - \frac{\dot{r}^2}{2c^2} + \frac{r\ddot{r}}{c^2}\right)$$

The velocity- and acceleration-dependent terms create an **effective
inertial mass** that depends on separation:

$$\mu_{\text{eff}}(r) = \mu\left(1 - \frac{\rho}{r}\right)$$

where $\mu$ is the reduced mass and $\rho = q_i q_j / (\mu c^2)$ is
the **critical radius**.

### Critical Radius and Sub-Critical Binding

For two like charges ($q_i q_j > 0$), the critical radius $\rho > 0$ is
a real positive distance. The effective inertial mass changes sign at
$r = \rho$, creating two permanently separated dynamical regimes:

- **Distant state** ($r > \rho$): ordinary Coulomb-like repulsion and
  scattering.
- **Molecular state** ($r < \rho$): the particles are bound, oscillating
  between their initial separation $r_0$ and $r = 0$. The sign reversal
  of $\mu_{\text{eff}}$ turns repulsion into effective attraction.

No continuous trajectory can cross $r = \rho$. Weber called this a
"molecular movement" (Sixth Memoir, §9.9).

### Why Collinear: The $\ell = 0$ Requirement

Sub-critical bound orbits exist **only** for head-on ($\ell = 0$)
collisions. For $\ell \neq 0$, the particles spiral to $r = 0$ at
infinite speed — a singularity that is not regularizable (Frauenfelder
& Weber 2024, Theorem 2.1). This forces all interacting pairs to
maintain zero angular momentum, which is automatically satisfied when
all particles lie on a line: every pair separation vector is parallel
to the line of motion.

### The Collinear Chain Configuration

Four equal positive charges are placed symmetrically on the x-axis:

```
    ←   1   ←   2   →   3   →   4   →
   -a       -b       +b       +a
```

The inner pair (2, 3) and outer pair (1, 4) oscillate in opposition,
while the "shoulder" pairs (1, 2) and (3, 4) bounce at close range.
All 6 pair separations are sub-critical:

| Pair | Particles | Initial $r$ | $\rho$ | Status |
|------|-----------|------------|--------|--------|
| Adjacent | (1,2), (3,4) | 0.030 | 0.125 | Sub-critical |
| Inner | (2,3) | 0.060 | 0.125 | Sub-critical |
| Cross | (1,3), (2,4) | 0.090 | 0.125 | Sub-critical |
| Outer | (1,4) | 0.120 | 0.125 | Sub-critical |

The outermost pair (1, 4) at $r = 0.120$ is just barely below the
critical radius $\rho = 0.125$.

### Why 4-Body Succeeds Where 3-Body Fails

Three collinear positive charges are unstable: a transverse perturbation
of $\epsilon = 10^{-10}$ grows by a factor of $10^8$ within $t < 5$.
The 4-body chain is stabilized by the symmetric shoulder forces — each
inner particle is flanked by two neighbors whose transverse force
components cancel by mirror symmetry. The outer particles, in turn, are
pushed inward by the three-body stack behind them. This mutual
restoring geometry keeps transverse perturbations bounded.

A tiny y-perturbation ($\epsilon = 10^{-3}$) is applied to particle 2
in this notebook to demonstrate transverse stability: the perturbation
oscillates but remains bounded throughout the simulation.

### Collision Bounce

When two particles approach within the bounce radius $r_{\text{bounce}}$,
their relative position is reflected: $\vec{q}_{\text{rel}} \to
-\vec{q}_{\text{rel}}$. This non-symplectic perturbation models the
$C^0$ continuation of head-on collisions through $r = 0$. Combined
with the unregularized symplectic integrator, energy errors remain
bounded.

**References**: Weber, Sixth Memoir (1871) §§9.8–9.17; Frauenfelder &
Weber, *Anal. Math. Phys.* **14**:31 (2024); see
`docs/theory/CriticalRadiusAndLikeChargeAttraction.md` and
`docs/exploratory/FourPositiveChargeCrossInvestigation.md`.

In [1]:
using WeberElectrodynamics
using LinearAlgebra
using Printf
using GLMakie  # or CairoMakie, WGLMakie

## 1. System Construction and Physical Parameters

In [2]:
# Physical parameters
m = 1.0               # equal masses
q = 1.0               # all positive charges
c = 4.0

# Derived quantities
mu = m * m / (m + m)              # reduced mass for any pair = 0.5
rho = q^2 / (mu * c^2)           # critical radius = 0.125

# Chain geometry
a = 0.06              # outer particle half-separation
b = 0.03              # inner particle half-separation

# All 6 pair distances
pair_info = [
    ("(1,2) adjacent",  a - b),
    ("(1,3) cross",     a + b),
    ("(1,4) outer",     2a),
    ("(2,3) inner",     2b),
    ("(2,4) cross",     a + b),
    ("(3,4) adjacent",  a - b)
]

# Integration parameters
dt = 1e-5
bounce_r = 0.048

system = WeberSystem(4, 2)

@printf("Four-body collinear chain:\n")
@printf("  Particles:  %d (2D)\n", system.n_particles)
@printf("  DOF:        %d\n", system.degrees_of_freedom)
@printf("\nChain geometry:  1 ——— 2 ——— 3 ——— 4\n")
@printf("Positions:     -a=-%.2f  -b=-%.2f  +b=+%.2f  +a=+%.2f\n", a, b, b, a)
@printf("\nCritical radius rho = %.4f (m=%.1f, q=+%.1f, c=%.1f)\n", rho, m, q, c)
@printf("\nAll 6 pair separations:\n")
for (label, d) in pair_info
    status = d < rho ? "sub-critical" : "SUPER-critical"
    margin = (rho - d) / rho * 100
    @printf("  %-16s  r = %.4f  (%+.1f%% margin)  %s\n", label, d, margin, status)
end
@printf("\nIntegration:\n")
@printf("  dt = %.0e, bounce_r = %.3f, c = %.1f\n", dt, bounce_r, c)

Four-body collinear chain:
  Particles:  4 (2D)
  DOF:        8

Chain geometry:  1 ——— 2 ——— 3 ——— 4
Positions:     -a=-0.06  -b=-0.03  +b=+0.03  +a=+0.06

Critical radius rho = 0.1250 (m=1.0, q=+1.0, c=4.0)

All 6 pair separations:
  (1,2) adjacent    r = 0.0300  (+76.0% margin)  sub-critical
  (1,3) cross       r = 0.0900  (+28.0% margin)  sub-critical
  (1,4) outer       r = 0.1200  (+4.0% margin)  sub-critical
  (2,3) inner       r = 0.0600  (+52.0% margin)  sub-critical
  (2,4) cross       r = 0.0900  (+28.0% margin)  sub-critical
  (3,4) adjacent    r = 0.0300  (+76.0% margin)  sub-critical

Integration:
  dt = 1e-05, bounce_r = 0.048, c = 4.0


## 2. Initial Conditions and Problem Setup

In [3]:
# Positions: 4 particles on x-axis at [-a, -b, +b, +a]
# Particle 2 gets a tiny y-perturbation to demonstrate 2D stability
epsilon = 1e-3

q0 = [-a, 0.0,       # particle 1: (-0.06, 0)
      -b, epsilon,    # particle 2: (-0.03, 0.001) — perturbed
      +b, 0.0,        # particle 3: (+0.03, 0)
      +a, 0.0]        # particle 4: (+0.06, 0)

# All particles start at rest
p0 = zeros(8)

tmax = Inf

prob = WeberProblem(system, (0.0, tmax), q0, p0;
    masses = [m, m, m, m], charges = [q, q, q, q], c = c, dt = dt,
    regularization_enabled = false,
    regularization_collision_bounce_radius = bounce_r,
    zollner_enabled = false, zollner_a = 0.0)

@printf("Initial conditions:\n")
for i in 1:4
    xi = q0[2i-1]
    yi = q0[2i]
    @printf("  Particle %d: (%.4f, %.4f)\n", i, xi, yi)
end
@printf("\nAll momenta = 0 (start from rest)\n")
@printf("Transverse perturbation on particle 2: epsilon = %.0e\n", epsilon)
@printf("\nProblem: tspan = (0, Inf), bounce_r = %.3f\n", bounce_r)
@printf("Regularization: disabled (unregularized symplectic integrator)\n")
@printf("Zollner: disabled (all kappas = 1.0)\n")

Initial conditions:
  Particle 1: (-0.0600, 0.0000)
  Particle 2: (-0.0300, 0.0010)
  Particle 3: (0.0300, 0.0000)
  Particle 4: (0.0600, 0.0000)

All momenta = 0 (start from rest)
Transverse perturbation on particle 2: epsilon = 1e-03

Problem: tspan = (0, Inf), bounce_r = 0.048
Regularization: disabled (unregularized symplectic integrator)
Zollner: disabled (all kappas = 1.0)


## 3. Live Animation

The streaming animation viewer integrates the system in real time,
displaying rolling trajectories, energy, momentum, angular momentum,
and phase space. Use the **Speed** slider to control how many integration
steps are computed per frame.

**What to observe**:
- The four particles bounce back and forth along the x-axis, confined
  within a narrow region around the origin.
- The energy panel shows bounded oscillation — the non-symplectic bounce
  perturbation causes slow drift, but energy remains within ~10%.
- Particle 2's y-coordinate (from the $\epsilon$ perturbation) oscillates
  but remains small, demonstrating transverse stability.

In [4]:
animate_weber(prob; buffer_size = 5000, tail_length = 500, compute_batch = 1)

GLMakie.Screen(...)